# Trabajo Practico 5 - Medidas Electrónicas 1
**Docentes:**

*   Ing. Marinsek, Emiliano
*   Ing. Perdomo, Juan Manuel

**Alumnos:**

*   Ferrario, Franco Ezequiel
*   Encinas, Leandro
*   Marchesi, Matias
*   Quiroga, Bruno

**Curso:** R4052

**Año:** 2026


# Librerias y funciones a utilizar

In [9]:
# Librerias de pyhon a utilizar
import numpy as np
import math
import IPython
from IPython.display import display, Math
import pandas as pd
from typing import Any


def incert_A(mediciones):
  s = np.std(mediciones, ddof=1, )  # ddof=1 -> grados de libertad v = 1/n-1
  return s/math.sqrt(len(mediciones))

#Incertidumbre tipo B para modo frecuencimetro
def incert_B(estabilidad_BT, delta_N, N, mediciones):
    """
    Calcula la incertidumbre Tipo B para el modo FRECUENCÍMETRO.
    """
    f_media = np.mean(mediciones)
    
    #Incertidumbre relativa del oscilador/base de tiempo
    u_bt_rel = estabilidad_BT / np.sqrt(3)
    
    #Incertidumbre relativa por el error de +/- 1 cuenta
    u_cuentas_rel = delta_N / (np.sqrt(3) * N)
    
    # Combinación de ambas componentes (relativas)
    return np.sqrt(u_bt_rel**2 + u_cuentas_rel**2)

# Adaptación de Incertidumbre Tipo B para el modo Periodímetro
def incert_B_periodimetro(estabilidad_BT, delta_N, N, resolucion_display):
    """
    Calcula la incertidumbre Tipo B combinando el error de la Base de Tiempo
    y el error de digitalización/resolución (distribución rectangular).
    """
    # Componente por estabilidad de la Base de Tiempo
    uB_bt = T_media * (estabilidad_BT / np.sqrt(3))
    
    # Componente por resolución del display (error de cuantización)
    # Se modela con distribución rectangular sobre el paso de visualización
    uB_res = resolucion_display / (2 * np.sqrt(3))
    
    # Combinación de ambas componentes Tipo B
    return np.sqrt(uB_bt**2 + uB_res**2)


def incert_B_ConMedia(err_rel, nd, ctas, media):
  return (err_rel/100 + nd/ctas) * media / math.sqrt(3)

def incert_Total(uA, uB):
  return math.sqrt(uA**2 + uB**2)

def incert_Comb_Rel(uX, uY,correlacion=0):
  return math.sqrt((uX['u_C']/uX['media'])**2 + (uY['u_C']/uY['media'])**2 - (2*(uX['u_C']/uX['media'])*(uY['u_C']/uY['media'])*correlacion))

# Generalidades de los instrumentos

El frecuencímetro utilizado para las prácticas es el **Protek U2000A**.

Para el cálculo de incertidumbres se utilizará la estabilidad de largo plazo de la
base de tiempo especificada por el fabricante, cuyo valor es:

$$
\frac{\Delta f_{BT}}{f_{BT}} = \pm 3\cdot10^{-7}/\text{mes}
$$

Como no se dispone de información sobre la antigüedad del instrumento, se
adopta como hipótesis una antigüedad de **5 años**, equivalentes a 60 meses.
Por lo tanto:

$$\frac{\Delta f_{BT}}{f_{BT}} = 3\cdot10^{-7}\cdot\frac{1}{\text{mes}}\cdot60\text{meses}$$

Expresando este valor en partes por millón:

$$
\frac{\Delta f_{BT}}{f_{BT}} = 18\,\text{ppm}
$$

y en porcentaje:

$$\frac{\Delta f_{BT}}{f_{BT}} = 18\cdot10^{-6} \cdot 100\% = 0.0018\%$$

Este valor será utilizado como error relativo de la base de tiempo en los
cálculos de incertidumbre.

A continuación, se coloca una celda con variables generales del instrumento que utilizaremos en todas las otras celdas.

In [5]:
k = 2

f_BT = 10e6                 # Frecuencia de la base de tiempo [Hz]

estabilidad_BT = 3e-7       # Estabilidad de largo plazo [1/mes]
antiguedad_meses = 5 * 12   # Antigüedad estimada del instrumento [meses]

error_rel_BT = estabilidad_BT * antiguedad_meses
error_rel_BT_pct = error_rel_BT * 100

nd = 1                      # Error de última cuenta/dígito

print(f"Base de tiempo: {f_BT/1e6} MHz")
print(f"Antigüedad estimada: {antiguedad_meses} meses")
print(f"Estabilidad acumulada: {error_rel_BT:.2e}")
print(f"Error relativo de BT: {error_rel_BT_pct:.4f} %")

Base de tiempo: 10.0 MHz
Antigüedad estimada: 60 meses
Estabilidad acumulada: 1.80e-05
Error relativo de BT: 0.0018 %


# Ejercicio 1

Las ecuaciones que se usarán para este ejercicio son las vistas en la cátedra

## Medición con frecuencia

Para la medición de frecuencia se usarán las ecuaciones:

$$f_I = 10^k \cdot f_{BT} \cdot N$$

Y su incertidumbre relativa:

$$\frac{\Delta f_I}{f_I} = \pm (\frac{\Delta f_{BT}}{f_{BT}} + \frac{\Delta N}{N})$$

Siendo el $\Delta N = 1$.

La cantidad de ciclos durante el gate time es:
$$N = f_x \cdot T_C$$.

Una vez obtenidos los datos se calcularán las incertidumbres.

## Medición con periodos promediados

Para la medición con periodos promediados se usarán las ecuaciones:
$$T_I = T_C \cdot N$$

Y su incertidumbre relativa:
$$\frac{\Delta T_I}{T_I} = \pm (\frac{\Delta f_{BT}}{f_{BT}} + \frac{\Delta N}{N} + \frac{e_D}{10^M})$$

Donde M es la cantidad de periodos promediados.
Además $e_D = 0$ porque estamos estudiando señales cuadradas, por lo que la incertdiumbre termina quedando:
$$\frac{\Delta T_I}{T_I} = \pm (\frac{\Delta f_{BT}}{f_{BT}} + \frac{\Delta N}{N})$$

## 10 Hz

Se medirá una señal cuadrada de $10Hz$ con $2V_{pp}$.

### 0.1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 0.1s de Gate time.



In [10]:
# Datos 0.1s Gate Time
f_x = 10          # Hz
gate_time = 0.1   # s
delta_N = 1

mediciones = [
    0.01e3, 0.01e3, 0.01e3, 0.01e3, 0.01e3,
    0.01e3, 0.01e3, 0.01e3, 0.01e3, 0.01e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.2f} Hz")
print(f"uA = {uA_f:.4f} Hz")
print(f"uB = {uB_f:.4f} Hz")
print(f"uC = {uC_f:.4f} Hz")
print(f"U = {U_f:.4f} Hz")

Media = 10.00 Hz
uA = 0.0000 Hz
uB = 0.5774 Hz
uC = 0.5774 Hz
U = 1.1547 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (10.1 \pm 1.1)Hz$$
    </div>
</div>

### 1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1s de Gate time.

In [11]:
# Datos 1s Gate Time
f_x = 10          # Hz
gate_time = 1   # s
delta_N = 1

mediciones = [
    0.010e3, 0.010e3, 0.010e3, 0.010e3, 0.010e3,
    0.010e3, 0.010e3, 0.010e3, 0.010e3, 0.010e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.2f} Hz")
print(f"uA = {uA_f:.4f} Hz")
print(f"uB = {uB_f:.4f} Hz")
print(f"uC = {uC_f:.4f} Hz")
print(f"U = {U_f:.4f} Hz")

Media = 10.00 Hz
uA = 0.0000 Hz
uB = 0.0577 Hz
uC = 0.0577 Hz
U = 0.1155 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (10.0 \pm 1.2)Hz$$
    </div>
</div>

### 10s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10s de Gate time.

In [12]:
# Datos 10s Gate Time
f_x = 10          # Hz
gate_time = 10   # s
delta_N = 1

mediciones = [
    0.0100e3, 0.0100e3, 0.0100e3, 0.0100e3, 0.0100e3,
    0.0100e3, 0.0100e3, 0.0100e3, 0.0100e3, 0.0100e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.2f} Hz")
print(f"uA = {uA_f:.4f} Hz")
print(f"uB = {uB_f:.4f} Hz")
print(f"uC = {uC_f:.4f} Hz")
print(f"U = {U_f:.4f} Hz")

Media = 10.00 Hz
uA = 0.0000 Hz
uB = 0.0058 Hz
uC = 0.0058 Hz
U = 0.0115 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (10.000 \pm 0.015)Hz$$
    </div>
</div>

### 1 Ciclo de periodos promediados

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1 ciclo de periodos promediados.

In [45]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones (1 µs expresado en segundos)
mediciones = [
    100003.0e-6,100002.9e-6,100003.0e-6,100002.9e-6,100003.0e-6,100002.9e-6,100003.0e-6,100002.9e-6,100003.0e-6,100002.9e-6

] # s


T_media = np.mean(mediciones)
M = 1 #Numero de ciclos

# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T

# IMPRESIÓN DE RESULTADOS
print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 1.0000295000e-01 s
uA = 1.66667e-08 s
uB = 2.89194e-07 s
uC = 2.89674e-07 s
U  = 5.79348e-07 s


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$P = (100002.95 \pm 0.58) μs$$
    </div>
</div>

### 10 ciclo de periodos promediados
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10 ciclo de periodos promediados.

In [44]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones (1 µs expresado en segundos)
mediciones = [
100003.06e-6,
100003.09e-6,
100003.13e-6,
100003.16e-6,
100003.24e-6,
100003.28e-6,
100003.27e-6,
100003.29e-6,
100003.30e-6,
100003.29e-6
] # s

T_media = np.mean(mediciones)
M = 10 #Numero de ciclos
# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T


# IMPRESIÓN DE RESULTADOS

print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 1.0000321100e-01 s
uA = 2.90765e-08 s
uB = 2.89194e-07 s
uC = 2.90652e-07 s
U  = 5.81305e-07 s


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$P = (100003.21 \pm 0.58) μs$$
    </div>
</div>

### 100 ciclo de periodos promediados
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 100 ciclo de periodos promediados.

In [43]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones  (1 µs expresado en segundos)
mediciones = [
0
] # s

T_media = np.mean(mediciones)
M = 100 #Numero de ciclos
# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T


# IMPRESIÓN DE RESULTADOS

print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 0.0000000000e+00 s
uA = nan s
uB = 2.88675e-07 s
uC = nan s
U  = nan s


<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        No pudimos realizar la medición ya que el instrumento falló por overflow
    </div>
</div>

## 1 KHz

### 0.1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 0.1s de Gate time.



In [29]:
# Datos 0.1s Gate Time
f_x = 1e3          # Hz
gate_time = 0.1   # s
delta_N = 1

mediciones = [
    1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.2f} Hz")
print(f"uA = {uA_f:.4f} Hz")
print(f"uB = {uB_f:.4f} Hz")
print(f"uC = {uC_f:.4f} Hz")
print(f"U = {U_f:.4f} Hz")

Media = 1000.00 Hz
uA = 0.0000 Hz
uB = 0.0058 Hz
uC = 0.0058 Hz
U = 0.0115 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (1000.000 \pm 0.011)Hz$$
    </div>
</div>

### 1s de Gate Time
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1s de Gate time.

In [32]:
# Datos 0.1s Gate Time
f_x = 1e3          # Hz
gate_time = 1   # s
delta_N = 1

mediciones = [
    1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.4f} Hz")
print(f"uA = {uA_f:.4f} Hz")
print(f"uB = {uB_f:.4f} Hz")
print(f"uC = {uC_f:.4f} Hz")
print(f"U = {U_f:.4f} Hz")

Media = 1000.0000 Hz
uA = 0.0000 Hz
uB = 0.0006 Hz
uC = 0.0006 Hz
U = 0.0012 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (1000.0000 \pm 0.0012)Hz$$
    </div>
</div>

### 10s de Gate Time
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10s de Gate time.

In [35]:
# Datos 0.1s Gate Time
f_x = 1e3          # Hz
gate_time = 10   # s
delta_N = 1

mediciones = [
    1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3,1e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.5f} Hz")
print(f"uA = {uA_f:.5f} Hz")
print(f"uB = {uB_f:.5f} Hz")
print(f"uC = {uC_f:.5f} Hz")
print(f"U = {U_f:.5f} Hz")

Media = 1000.00000 Hz
uA = 0.00000 Hz
uB = 0.00006 Hz
uC = 0.00006 Hz
U = 0.00012 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (1000.00000 \pm 0.00012)Hz$$
    </div>
</div>

### 1 ciclo de periodos promediados
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1 ciclo de periodos promediados.

In [42]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones(1 µs expresado en segundos)
mediciones = [
   1000.0e-6,1000.1e-6,1000.0e-6,1000.1e-6,1000.0e-6,1000.1e-6,1000.0e-6,1000.1e-6,1000.0e-6,1000.1e-6
] # s


T_media = np.mean(mediciones)
M = 1 #Numero de ciclos

# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T

# IMPRESIÓN DE RESULTADOS
print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 1.0000500000e-03 s
uA = 1.66667e-08 s
uB = 2.88675e-07 s
uC = 2.89156e-07 s
U  = 5.78312e-07 s


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$P = (1000.05 \pm 0.58) μs$$
    </div>
</div>

### 10 ciclo de periodos promediados
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10 ciclo de periodos promediados.

In [46]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones(1 µs expresado en segundos)
mediciones = [
1000.03e-6,
1000.04e-6,
1000.03e-6,
1000.04e-6,
1000.03e-6,
1000.04e-6,
1000.03e-6,
1000.04e-6,
1000.03e-6,
1000.04e-6
] # s


T_media = np.mean(mediciones)
M = 10 #Numero de ciclos

# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T

# IMPRESIÓN DE RESULTADOS
print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 1.0000350000e-03 s
uA = 1.66667e-09 s
uB = 2.88675e-07 s
uC = 2.88680e-07 s
U  = 5.77360e-07 s


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$P = (1000.03 \pm 0.58) μs$$
    </div>
</div>

### 100 ciclo de periodos promediados
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 100 ciclo de periodos promediados.

In [47]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones(1 µs expresado en segundos)
mediciones = [
1000.035e-6,
1000.036e-6,
1000.035e-6,
1000.036e-6,
1000.035e-6,
1000.036e-6,
1000.035e-6,
1000.036e-6,
1000.035e-6,
1000.036e-6
] # s


T_media = np.mean(mediciones)
M = 100 #Numero de ciclos

# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T

# IMPRESIÓN DE RESULTADOS
print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 1.0000355000e-03 s
uA = 1.66667e-10 s
uB = 2.88675e-07 s
uC = 2.88675e-07 s
U  = 5.77350e-07 s


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$P = (1000.03 \pm 0.58) μs$$
    </div>
</div>

### 1MHz

### 0.1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 0.1s de Gate time.



In [49]:
# Datos 0.1s Gate Time
f_x = 1e6          # Hz
gate_time = 0.1   # s
delta_N = 1

mediciones = [
    999.96e3,999.97e3,999.96e3,999.97e3,999.96e3,999.97e3,999.96e3,999.97e3,999.96e3,999.97e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.4f} Hz")
print(f"uA = {uA_f:.4f} Hz")
print(f"uB = {uB_f:.4f} Hz")
print(f"uC = {uC_f:.4f} Hz")
print(f"U = {U_f:.4f} Hz")


Media = 999965.0000 Hz
uA = 1.6667 Hz
uB = 0.0000 Hz
uC = 1.6667 Hz
U = 3.3333 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (999965.0 \pm 3.3)Hz$$
    </div>
</div>

### 1s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1s de Gate time.



In [53]:
# Datos 1s Gate Time
f_x = 1e6          # Hz
gate_time = 1   # s
delta_N = 1

mediciones = [
999.965e3,
999.964e3,
999.964e3,
999.965e3,
999.965e3,
999.965e3,
999.965e3,
999.965e3,
999.964e3,
999.964e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.4f} Hz")
print(f"uA = {uA_f:.4f} Hz")
print(f"uB = {uB_f:.4f} Hz")
print(f"uC = {uC_f:.4f} Hz")
print(f"U = {U_f:.4f} Hz")


Media = 999964.6000 Hz
uA = 0.1633 Hz
uB = 0.0000 Hz
uC = 0.1633 Hz
U = 0.3266 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (999964.60 \pm 0.33)Hz$$
    </div>
</div>

### 10s de Gate Time

Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10s de Gate time.



In [55]:
# Datos 1s Gate Time
f_x = 1e6          # Hz
gate_time = 10   # s
delta_N = 1

mediciones = [
999.9648e3,
999.9648e3,
999.9648e3,
999.9647e3,
999.9648e3,
999.9648e3,
999.9648e3,
999.9647e3,
999.9648e3,
999.9648e3
]  # Hz

f_media = np.mean(mediciones)

# Cantidad de cuentas durante el gate time
N = f_media * gate_time

# Incertidumbre Tipo A
uA_f = incert_A(mediciones)

# Incertidumbre Tipo B

uB_f = incert_B(
    estabilidad_BT,
    delta_N,
    N,
    mediciones
)

# Incertidumbre total
uC_f = incert_Total(uA_f, uB_f)

U_f = k * uC_f

print(f"Media = {f_media:.4f} Hz")
print(f"uA = {uA_f:.4f} Hz")
print(f"uB = {uB_f:.4f} Hz")
print(f"uC = {uC_f:.4f} Hz")
print(f"U = {U_f:.4f} Hz")


Media = 999964.7800 Hz
uA = 0.0133 Hz
uB = 0.0000 Hz
uC = 0.0133 Hz
U = 0.0267 Hz


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$f = (999964.780 \pm 0.027)Hz$$
    </div>
</div>

### 1 ciclo de periodos promediados
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 1 ciclo de periodos promediados.

In [58]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones(1 µs expresado en segundos)
mediciones = [
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6
] # s


T_media = np.mean(mediciones)
M = 1 #Numero de ciclos

# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T

# IMPRESIÓN DE RESULTADOS
print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 1.0000000000e-06 s
uA = 7.05861e-23 s
uB = 2.88675e-07 s
uC = 2.88675e-07 s
U  = 5.77350e-07 s


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$P = (1000.00 \pm 0.57) μs$$
    </div>
</div>

### 10 ciclo de periodos promediados
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 10 ciclo de periodos promediados.

In [60]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones(1 µs expresado en segundos)
mediciones = [
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6
] # s


T_media = np.mean(mediciones)
M = 10 #Numero de ciclos

# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T

# IMPRESIÓN DE RESULTADOS
print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 1.0000000000e-06 s
uA = 7.05861e-23 s
uB = 2.88675e-07 s
uC = 2.88675e-07 s
U  = 5.77350e-07 s


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$P = (1000.00 \pm 0.58) μs$$
    </div>
</div>

### 100 ciclo de periodos promediados
Partiendo de las mediciones hechas en el laboratorio se harán las cuentas para 100 ciclo de periodos promediados.

In [62]:
# ==========================================
# CONFIGURACIÓN DE DATOS (MODO PERIODÍMETRO)
# ==========================================

# Mediciones(1 µs expresado en segundos)
mediciones = [
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6,
1e-6
] # s


T_media = np.mean(mediciones)
M = 100 #Numero de ciclos

# En periodimetros, N representa las cuentas del reloj interno durante 1 ciclo de la señal
# N = T_media / T_clock = T_media * f_clock
N = M * T_media * f_BT

# Paso mínimo visible de tus datos actuales (para evaluar el display)
resolucion_display = 1e-6  # 1 µs

# CÁLCULO DE INCERTIDUMBRES
# Incertidumbre Tipo A
uA_T = incert_A(mediciones)

# Incertidumbre Tipo B (Adaptada a periodo)
uB_T = incert_B_periodimetro(estabilidad_BT, nd, N, resolucion_display)

# Incertidumbre total combinada
uC_T = incert_Total(uA_T, uB_T)

# Incertidumbre expandida
U_T = k * uC_T

# IMPRESIÓN DE RESULTADOS
print(f"Media = {T_media:.10e} s")
print(f"uA = {uA_T:.5e} s")
print(f"uB = {uB_T:.5e} s")
print(f"uC = {uC_T:.5e} s")
print(f"U  = {U_T:.5e} s")

Media = 1.0000000000e-06 s
uA = 7.05861e-23 s
uB = 2.88675e-07 s
uC = 2.88675e-07 s
U  = 5.77350e-07 s


Finalmente se expresa el resultado:

<div style="text-align:center;"">
    <div style="border: 2px solid #555; padding: 5px; border-radius: 8px; text-align: center; background-color: #f9f9f9; display: inline-block;">
        $$P = (1000.00 \pm 0.58) μs$$
    </div>
</div>